# Confronto europeo

Confronti armonizzati Eurostat. Il notebook usa la sezione `confronto_europeo` e permette di controllare pressione fiscale, spesa pubblica e spesa sociale.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_JSON = PROJECT_ROOT / "data/export/bilancio-pubblico/source-data.json"
SECTION_DIR = PROJECT_ROOT / "data/export/bilancio-pubblico/sections"

def load_section(section_id):
    section_path = SECTION_DIR / f"{section_id}.json"
    if section_path.exists():
        payload = json.loads(section_path.read_text(encoding="utf-8"))
        return payload.get("section", payload)
    payload = json.loads(SOURCE_JSON.read_text(encoding="utf-8"))
    return payload["sections"][section_id]

def frame(rows):
    return pd.DataFrame(rows or [])

def plot_bar(data, x, y, title, top=15):
    df = data.dropna(subset=[y]).sort_values(y, ascending=False).head(top).sort_values(y)
    ax = df.plot.barh(x=x, y=y, legend=False, figsize=(9, max(4, len(df) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(y)
    ax.set_ylabel("")
    plt.tight_layout()
    return ax

def plot_line(data, x, y, title):
    df = data.dropna(subset=[x, y]).sort_values(x)
    ax = df.plot.line(x=x, y=y, marker="o", legend=False, figsize=(9, 4.8))
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.tight_layout()
    return ax

europa = load_section("confronto_europeo")
europa.keys()

## Metriche disponibili

Le metriche principali sono costruite sulle stesse righe paese.

In [ ]:
metrics = pd.DataFrame(europa["metrics"])
display(metrics)

## Paesi europei

La tabella combina pressione fiscale, spesa pubblica e spesa sociale. I valori sono in percentuale del PIL.

In [ ]:
peer = frame(europa["peer"])
cols = ["country", "code", "tax_pressure", "tax_year", "public_spending", "spending_year", "social_spending", "social_year"]
display(peer[[c for c in cols if c in peer.columns]].sort_values("country").head(30))

## Pressione fiscale

Classifica dei paesi disponibili per pressione fiscale e contributiva.

In [ ]:
plot_bar(peer.rename(columns={"tax_pressure": "percent_pil"}), "country", "percent_pil", "Pressione fiscale e contributiva in Europa")

## Spesa pubblica

Classifica dei paesi disponibili per spesa pubblica totale.

In [ ]:
plot_bar(peer.rename(columns={"public_spending": "percent_pil"}), "country", "percent_pil", "Spesa pubblica totale in Europa")

## Dettaglio COFOG europeo

Il dettaglio per funzione permette confronti più granulari, usando le opzioni disponibili nell'export.

In [ ]:
options = frame(europa["peer_spending_function_options"])
display(options.head(30))
functions = frame(europa["peer_spending_functions"])
display(functions.head())